# Data preparation and filtering of original dataset to use for the Neural Network

Transparency is important so I'm showing all my steps for the data preparation. The original dataset with images is too large to include in the GitHub, so you need to download it yourself. Make sure you store it somewhere where you can access it via a path.


In [1]:
import pandas as pd
from pathlib import Path
import json 
import numpy as np
import cv2
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupKFold
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import random

# check for model running: 
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.12.0+cpu
None
False


AssertionError: Torch not compiled with CUDA enabled

In [2]:
# -- IMPORTING ALL NON-IMAGE DATA -- #
# ground truth segmentations per img
seg = pd.read_csv("Radiology_hand_drawn_segmentations_v2.csv")
# anottations per img
annot = pd.read_excel("Radiology-manual-annotations.xlsx")

# The segmentation is sorted by the column called #filename. Annot is sorted by the column Image_name
# The only difference about these two is that #filename has .jpg after each entry. 
# So i'm removing that 

seg["#filename"] = seg["#filename"].str.replace(".jpg", "")

# Analysing values in seg:
# Using print(seg['COLUMNNAME'].unique()) I figured out that the columns region_attributes and file_attributes only contain {}. 
# I also don't think that the column "file_size" will be of any use. We can always change this if one of them is needed later.
seg = seg.drop(columns = ["file_attributes", "region_attributes", "file_size"])

# -- IMPORTING IMAGE DATA -- #
# Make sure you have the images dataset folder downloaded and saved in program files.
# Or change the path stuff around. 

# Due to python tweaking if you put in the raw path files with \, 
# put r in front of the string to treat it as raw string
imgs_locations = [Path(r"C:\Program Files\PKG - CDD-CESM\CDD-CESM\Low energy images of CDD-CESM"),
                  Path(r"C:\Program Files\PKG - CDD-CESM\CDD-CESM\Subtracted images of CDD-CESM")]

print(seg)
print(annot)
print(imgs_locations)
rows = []
for folder in imgs_locations:
    for img in folder.glob("*.jpg"):  # only get images and not the desktop.ini file
        rows.append({"image_id": img.stem, "image_path": str(img)})
images_df = pd.DataFrame(rows)

# Time for the full dataset? Yeahhh
dataset = (images_df
           .merge(annot, left_on = "image_id", right_on = "Image_name", how = "left")
           .merge(seg, left_on = "image_id", right_on = "#filename", how = "left"))

dataset["patient_id"] = dataset["image_id"].str.extract(r"(P\d+)")
dataset = dataset.drop(columns = ["Patient_ID", "Image_name"])
dataset


          #filename  region_count  region_id  \
0       P1_L_CM_MLO             1          0   
1       P1_L_DM_MLO             1          0   
2        P2_L_CM_CC             1          0   
3       P2_L_CM_MLO             1          0   
4        P2_L_DM_CC             4          0   
...             ...           ...        ...   
2966  P326_L_CM_MLO             1          0   
2967   P326_L_DM_CC             1          0   
2968  P326_L_DM_MLO             1          0   
2969   P326_R_DM_CC             1          0   
2970  P326_R_DM_MLO             1          0   

                                region_shape_attributes  
0     {"name":"polygon","all_points_x":[350,423,1005...  
1     {"name":"polygon","all_points_x":[353,413,1033...  
2     {"name":"polygon","all_points_x":[550,533,412,...  
3     {"name":"polygon","all_points_x":[26,954,945,8...  
4     {"name":"polygon","all_points_x":[4,107,349,49...  
...                                                 ...  
2966  {"name":"po

,image_id,image_path,Side,Type,Age,Breast density (ACR),BIRADS,Findings,View,Tags,Machine,Pathology Classification/ Follow up,#filename,region_count,region_id,region_shape_attributes,patient_id
0,P100_L_DM_CC,C:\Program Files\PKG - CDD-CESM\CDD-CESM\Low e...,L,DM,61.0,_,2,Edema and postoperative scar,CC,postoperative,1.0,Benign,P100_L_DM_CC,2.0,0.0,"{""name"":""polygon"",""all_points_x"":[52,396,714,9...",P100
1,P100_L_DM_CC,C:\Program Files\PKG - CDD-CESM\CDD-CESM\Low e...,L,DM,61.0,_,2,Edema and postoperative scar,CC,postoperative,1.0,Benign,P100_L_DM_CC,2.0,1.0,"{""name"":""polygon"",""all_points_x"":[187,258,202,...",P100
2,P100_L_DM_MLO,C:\Program Files\PKG - CDD-CESM\CDD-CESM\Low e...,L,DM,61.0,_,2,Edema and postoperative scar,MLO,postoperative,1.0,Benign,P100_L_DM_MLO,2.0,0.0,"{""name"":""polygon"",""all_points_x"":[20,233,569,8...",P100
3,P100_L_DM_MLO,C:\Program Files\PKG - CDD-CESM\CDD-CESM\Low e...,L,DM,61.0,_,2,Edema and postoperative scar,MLO,postoperative,1.0,Benign,P100_L_DM_MLO,2.0,1.0,"{""name"":""polygon"",""all_points_x"":[175,363,531,...",P100
4,P100_R_DM_CC,C:\Program Files\PKG - CDD-CESM\CDD-CESM\Low e...,R,DM,61.0,C,1,Normal,CC,normal,1.0,Normal,NaN,NaN,NaN,NaN,P100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3739,P99_L_CM_MLO,C:\Program Files\PKG - CDD-CESM\CDD-CESM\Subtr...,L,CESM,61.0,_,1,No mass or non mass enhancement,MLO,normal,1.0,Normal,NaN,NaN,NaN,NaN,P99
3740,P99_L_CM_MLO_2,C:\Program Files\PKG - CDD-CESM\CDD-CESM\Subtr...,L,CESM,61.0,_,1,No mass or non mass enhancement,MLO,normal,1.0,Normal,NaN,NaN,NaN,NaN,P99
3741,P99_R_CM_CC,C:\Program Files\PKG - CDD-CESM\CDD-CESM\Subtr...,R,CESM,61.0,_,5,Heterogenously enhancing mass with irregular m...,CC,"malignant mass, heterogenous, irregular",1.0,Malignant,P99_R_CM_CC,1.0,0.0,"{""name"":""polygon"",""all_points_x"":[2376,2326,22...",P99
3742,P99_R_CM_MLO,C:\Program Files\PKG - CDD-CESM\CDD-CESM\Subtr...,R,CESM,61.0,_,5,Heterogenously enhancing mass with irregular m...,MLO,"malignant mass, heterogenous, irregular",1.0,Malignant,P99_R_CM_MLO,1.0,0.0,"{""name"":""polygon"",""all_points_x"":[1656,1693,17...",P99


In [3]:
# -- Mask time for ground truth -- #

def create_mask(image_path, annot):
    """
    Creates a binary mask based on region shape attributes for an image, to use as truth.
    """
    img = np.array(Image.open(image_path))
    height, width = img.shape[:2]

    # mask should contain all 0's with 1 for identified tumors 
    mask = np.zeros((height, width), dtype=np.uint8) 

    for annotation in annot.dropna():
        shape = json.loads(annotation) # annotation is currently json so no more

        # -- We gotta keep the different shapes of annotations in mind! -- #
        # Seems like polygon, ellipse and circle are the ones used.
        shape_type = shape.get("name")

        if shape_type == "polygon":
            points = np.array(
                list(zip(shape["all_points_x"], shape["all_points_y"])),dtype=np.int32)
            cv2.fillPoly(mask, [points], 1)  # cv fillpoly is made for the polys

        elif shape_type == "circle":
            cv2.circle(mask,
                (shape["cx"], shape["cy"]), shape["r"],1, -1) # we can change color & thickness to outlines only, but i dont think we should

        elif shape_type == "ellipse":
            cv2.ellipse(
                mask,(shape["cx"], shape["cy"]),(shape["rx"], shape["ry"]), 0, 0,360, 1, -1)

    return mask

# -- Creating a new folder with all the masks per scan -- #

mask_folder = Path("masks")
mask_folder.mkdir(exist_ok=True)
grouped = dataset.groupby("image_id")
for image_id, group in grouped:

    if group["region_shape_attributes"].isna().all():
        continue

    image_path = group["image_path"].iloc[0]
    mask = create_mask(image_path, group["region_shape_attributes"])

    cv2.imwrite(str(mask_folder / f"{image_id}_mask.jpg"), mask * 255)


KeyboardInterrupt: 

In [4]:
# To prevent errors later on during training, the images without masks need to be removed.

valid_ids = []

for image_id in dataset["image_id"].unique():
    if Path(f"masks/{image_id}_mask.jpg").exists():
        valid_ids.append(image_id)

dataset = dataset[dataset["image_id"].isin(valid_ids)]

# Splitting
**Train** 70%

**Validation** 15%

**Test** 15%

In [5]:
patients = dataset["patient_id"].unique()
train_patients, test_patients = train_test_split(
    patients,
    test_size=0.1,   
    random_state=42,
    shuffle=True
)
#build dataframes (use dataset, not df)
df_trainval = dataset[dataset["patient_id"].isin(train_patients)]
# Don't touch this till the end
df_test = dataset[dataset["patient_id"].isin(test_patients)]
# safety check (so hopefully leakage)
assert set(df_trainval.patient_id).isdisjoint(set(df_test.patient_id))

#sizes
print("Train/Val:", len(df_trainval))
print("Test:", len(df_test))

Train/Val: 2507
Test: 464


In [6]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets):

        preds = torch.sigmoid(preds)

        preds = preds.view(-1)
        targets = targets.view(-1)

        intersection = (preds * targets).sum()

        dice = (2. * intersection + self.smooth) / (
            preds.sum() + targets.sum() + self.smooth
        )

        return 1 - dice

class BreastDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img = cv2.imread(row["image_path"], cv2.IMREAD_GRAYSCALE)
        mask_path = f"masks/{row['image_id']}_mask.jpg"
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        target_size = (512, 512)

        img = cv2.resize(img,target_size, interpolation=cv2.INTER_AREA)
        mask = cv2.resize(mask, target_size, interpolation = cv2.INTER_NEAREST)

        if img is None or mask is None:
            raise ValueError("Missing image or mask")

        img = img.astype(np.float32) / 255.0
        mask = (mask > 0).astype(np.float32)

        img = np.expand_dims(img, 0)
        mask = np.expand_dims(mask, 0)

        return torch.tensor(img), torch.tensor(mask)


In [7]:
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()

        self.pool = nn.MaxPool2d(2)

        self.down1 = DoubleConv(in_channels, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)

        self.bottleneck = DoubleConv(512, 1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.conv4 = DoubleConv(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.conv3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.conv2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.conv1 = DoubleConv(128, 64)

        self.out = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):

        x1 = self.down1(x)
        x2 = self.down2(self.pool(x1))
        x3 = self.down3(self.pool(x2))
        x4 = self.down4(self.pool(x3))

        x5 = self.bottleneck(self.pool(x4))

        x = self.up4(x5)
        x = torch.cat([x, x4], dim=1)
        x = self.conv4(x)

        x = self.up3(x)
        x = torch.cat([x, x3], dim=1)
        x = self.conv3(x)

        x = self.up2(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv2(x)

        x = self.up1(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv1(x)

        return self.out(x)



In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

groups = df_trainval["patient_id"]
group_k_fold = GroupKFold(n_splits=5)
fold_scores = []
for fold, (train_idx, val_idx) in enumerate(group_k_fold.split(df_trainval, groups=groups)):

    print(f"FOLD {fold+1}")
    train_df = df_trainval.iloc[train_idx]
    val_df   = df_trainval.iloc[val_idx]

    assert set(train_df.patient_id).isdisjoint(set(val_df.patient_id))
    print("Train patients:", train_df["patient_id"].nunique())
    print("Val patients:", val_df["patient_id"].nunique())

    # datata
    train_ds = BreastDataset(train_df)
    val_ds   = BreastDataset(val_df)

    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=4)

    # model, new one per fold
    model = UNet().to(device)

    loss_fn = DiceLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val = float("inf")


FOLD 1
Train patients: 192
Val patients: 48
FOLD 2
Train patients: 192
Val patients: 48
FOLD 3
Train patients: 192
Val patients: 48
FOLD 4
Train patients: 192
Val patients: 48
FOLD 5
Train patients: 192
Val patients: 48


In [ ]:
  for epoch in range(10):

        # TRAIN
        model.train()
        train_loss = 0

        for imgs, masks in train_loader:
            imgs = imgs.to(device)
            masks = masks.to(device)

            preds = model(imgs)
            loss = loss_fn(preds, masks)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # evalidated? I hope it works? 
        model.eval()
        val_loss = 0

        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs = imgs.to(device)
                masks = masks.to(device)

                preds = model(imgs)
                loss = loss_fn(preds, masks)

                val_loss += loss.item()

        train_loss /= len(train_loader)
        val_loss /= len(val_loader)

        print(f"Epoch {epoch+1}: Train={train_loss:.4f} | Val={val_loss:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), f"unet_fold{fold+1}.pth")

    fold_scores.append(best_val)

In [12]:
# Results? I think? 
print("CROSS VALIDATION RESULTS")

print("Fold scores:", fold_scores)
print("Mean:", np.mean(fold_scores))
print("Std:", np.std(fold_scores))

CROSS VALIDATION RESULTS
Fold scores: []
Mean: nan
Std: nan


C:\Users\reyna\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\reyna\anaconda3\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
C:\Users\reyna\anaconda3\Lib\site-packages\numpy\core\_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\reyna\anaconda3\Lib\site-packages\numpy\core\_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
C:\Users\reyna\anaconda3\Lib\site-packages\numpy\core\_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [14]:
# Testing the 3 best folds. 
test_ds = BreastDataset(df_test)

test_loader = DataLoader(test_ds, batch_size=4, shuffle=False)

loss_fn = DiceLoss()

test_scores = {}

for fold in [2,3,4]:


    model = UNet().to(device)

    model.load_state_dict(torch.load(f"unet_fold{fold}.pth",map_location=device))

    model.eval()

    test_loss = 0.0

    print(f"Testing fold {fold}")

    with torch.no_grad():

        for imgs, masks in test_loader:

            imgs = imgs.to(device)
            masks = masks.to(device)
            preds = model(imgs)
            loss = loss_fn(preds, masks)
            test_loss += loss.item()

    test_loss /= len(test_loader)
    test_scores[fold] = test_loss
    print(f"Fold {fold}: Test Loss = {test_loss:.4f}")

print("Results from test")

for fold, score in sorted(test_scores.items(), key=lambda x: x[1]):
    print(f"Fold {fold}: {score:.4f}")

best_fold = min(test_scores, key=test_scores.get)

print(f"\nBest fold on test set: Fold {best_fold}")
print(f"Test loss: {test_scores[best_fold]:.4f}")

Testing fold 2


KeyboardInterrupt: 

In [8]:
# visualisation of the model. 
idx = np.random.choice(len(test_ds))
print(f"Showing test image {idx}")
img, mask = test_ds[idx]

model.eval()

with torch.no_grad():
    logits = model(img.unsqueeze(0).to(device))
    dice_loss = loss_fn(logits, mask.unsqueeze(0).to(device)).item()

    pred = torch.sigmoid(logits)
    pred = (pred > 0.5).float()

img = img.squeeze().cpu()
mask = mask.squeeze().cpu()
pred = pred.squeeze().cpu()

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(img, cmap="gray")
plt.title("Image")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(gt, cmap="gray")
plt.title("Ground Truth")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(pred, cmap="gray")
plt.title(f"Prediction: Dice Loss = {dice_loss:.4f}")
plt.axis("off")

plt.show()

NameError: name 'test_ds' is not defined

# To do list:
* Implementeer meer metrics dan alleen dice loss: Jaccard index, accuracy, pixel accuracy, prec/rec/f1?
* Kies om maybe te focussen op 1 soort scan ipv allebei, gwn checken of t model dan beter is, OF 2 seperate modellen trainen.
* Alle scans maybe zorgen dat ze links/rechts gemirrored zijn? En bijbehorende ground truth dus ook.
* Zoeken naar een beter Unet model?

	
## Data Citation
Khaled R., Helal M., Alfarghaly O., Mokhtar O., Elkorany A., El Kassas H., Fahmy A. Categorized Digital Database for Low energy and Subtracted Contrast Enhanced Spectral Mammography images [Dataset]. (2021) The Cancer Imaging Archive. DOI:  10.7937/29kw-ae92 

Khaled, R., Helal, M., Alfarghaly, O., Mokhtar, O., Elkorany, A., El Kassas, H., & Fahmy, A. Categorized contrast enhanced mammography dataset for diagnostic and artificial intelligence research. (2022) Scientific Data, Volume 9, Issue 1. DOI: 10.1038/s41597-022-01238-0

Clark K, Vendt B, Smith K, Freymann J, Kirby J, Koppel P, Moore S, Phillips S, Maffitt D, Pringle M, Tarbox L, Prior F. The Cancer Imaging Archive (TCIA): Maintaining and Operating a Public Information Repository, Journal of Digital Imaging, Volume 26, Number 6, December, 2013, pp 1045-1057. DOI: 10.1007/s10278-013-9622-7